# HANCOCK Dataset — JSON to CSV Converter

Converts the 4 HANCOCK JSON files to CSV format:
- `blood_data.json` — 23,234 blood test records
- `clinical_data.json` — 763 patients, clinical info
- `pathological_data.json` — 763 patients, pathology info
- `blood_data_reference_ranges.json` — 38 analyte reference ranges

While the HANCOCK dataset includes longitudinal blood measurements, we exclude this modality from our framework for several reasons. Blood data in this dataset is sparse and unevenly sampled across patients, with measurements taken at varying time points relative to treatment. Incorporating it would require significant imputation and temporal aggregation and add complexity without a clear benefit to our target tasks. More importantly, HPV status and tumor subsite classification are primarily driven by histomorphological and anatomical features rather than hematological markers which makes blood data a weak signal for our specific prediction objectives.

In [1]:
import json
import csv
import os
import pandas as pd
from pathlib import Path

In [16]:
from google.colab import files
uploaded = files.upload()

Saving clinical_data.json to clinical_data (1).json
Saving pathological_data.json to pathological_data (1).json


In [17]:
INPUT_DIR = Path(".")
OUTPUT_DIR = Path("csv_output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = [
    "clinical_data.json",
    "pathological_data.json"
]


## Convert JSON → CSV

In [18]:
results = {}

for filename in FILES:
    json_path = INPUT_DIR / filename
    csv_path  = OUTPUT_DIR / (json_path.stem + ".csv")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data)
    df.to_csv(csv_path, index=False)
    results[filename] = {"rows": len(df), "columns": len(df.columns), "path": csv_path}
    print(f"[+] {filename:45s}  →  {len(df):>6,} rows × {len(df.columns)} cols  →  {csv_path}")

print("\nDone!")


[+] clinical_data.json                             →     763 rows × 32 cols  →  csv_output/clinical_data.csv
[+] pathological_data.json                         →     763 rows × 18 cols  →  csv_output/pathological_data.csv

Done!


## Selecting Samples

In [20]:
patient_ids = [484, 670, 36, 527, 125, 37, 607, 33, 650, 466, 384, 462, 269, 722, 314, 483, 284, 606, 642, 434, 24, 68, 510, 228, 208, 324, 662, 517, 81, 500, 234, 79, 189, 393, 7, 501, 211, 149, 695, 608, 61, 29, 216, 280, 392, 117, 183, 429, 174, 41, 140, 318, 168, 22, 10, 122, 144, 301, 502, 57, 262, 19, 255, 528, 604, 625, 630, 691, 757, 35, 615, 333, 693, 669, 171, 386, 539, 595, 351, 12, 315, 617, 651, 187, 312, 123, 655, 711, 647, 63, 136, 253, 102, 640, 631, 152, 247, 699, 261, 98, 130, 175, 649, 676, 66, 128, 325, 274, 111, 660, 626, 494, 388, 311, 243, 709, 374, 704, 688, 43, 703, 509, 179, 402, 418, 50, 65, 658, 192, 290, 46, 590, 320, 135, 157, 588, 394, 185, 613, 133, 295, 531, 233, 8, 464, 287, 104, 470, 584, 321, 358, 62, 59, 193, 346, 576, 614, 167, 542, 512, 162, 558, 403, 715, 44, 459, 349, 373, 732, 28, 74, 121, 661, 51, 113, 717, 6, 334, 212, 385, 616, 707, 736, 555, 644, 352, 292, 276, 439, 583, 298, 238, 151, 3, 30, 390, 664, 627, 83, 209, 410, 93, 72, 400, 559, 20, 38, 475, 16, 141, 214, 746, 399, 430, 153, 114, 159, 139, 712, 570, 415]

In [22]:
patient_ids_str = [str(p).zfill(3) for p in patient_ids]

df_clinical     = pd.read_csv("csv_output/clinical_data.csv")
df_pathological = pd.read_csv("csv_output/pathological_data.csv")

df_clinical_filtered     = df_clinical[df_clinical["patient_id"].astype(str).isin(patient_ids_str)]
df_pathological_filtered = df_pathological[df_pathological["patient_id"].astype(str).isin(patient_ids_str)]

df_clinical_filtered.to_csv("clinical_data_filtered.csv", index=False)
df_pathological_filtered.to_csv("pathological_data_filtered.csv", index=False)

print(f"Clinical:     {len(df_clinical)} → {len(df_clinical_filtered)} rows")
print(f"Pathological: {len(df_pathological)} → {len(df_pathological_filtered)} rows")

Clinical:     763 → 181 rows
Pathological: 763 → 181 rows
